# 08 · Complex types: structs, arrays, maps & JSON

Real-world data (especially JSON from APIs and event logs) is nested. Spark has three complex types:

| Type | Like | Example |
|---|---|---|
| `struct` | a nested row / Python object with fixed fields | `device: {os: "iOS", version: "17.1"}` |
| `array` | a Python list (all elements same type) | `tags: ["sale", "new"]` |
| `map` | a Python dict (keys same type, values same type) | `properties: {"page": "/home"}` |

In [ ]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [ ]:
events = spark.read.json(f"{DATA}/events.jsonl")
events.printSchema()
events.show(3, truncate=False)

## Structs
Reach inside with a dot. `"device.*"` expands all fields into top-level columns.

In [ ]:
events.select("user_id", "device.os", F.col("device.version").alias("os_version")).show(3)
events.select("user_id", "device.*").show(3)

## Arrays

In [ ]:
events.select(
    "tags",
    F.size("tags").alias("n_tags"),
    F.array_contains("tags", "sale").alias("has_sale"),
    F.col("tags")[0].alias("first_tag"),                 # 0-based; null if missing
    F.sort_array("tags").alias("sorted"),
).show(5, truncate=False)

### `explode`: one row per array element
This is how you "flatten" arrays to analyse their contents. `explode` **drops** rows whose array is empty
or null; `explode_outer` keeps them (with a null). `posexplode` also gives the position.

In [ ]:
small = events.select("user_id", "tags").limit(4)
small.show(truncate=False)
small.select("user_id", F.explode("tags").alias("tag")).show()
small.select("user_id", F.explode_outer("tags").alias("tag")).show()

## Maps
Spark's JSON reader inferred `properties` as a *struct*. If the keys vary from record to record, a **map**
is the better type — and we can request it with an explicit schema:

In [ ]:
schema = """
    user_id int, ts timestamp, event string,
    device struct<os: string, version: string>,
    tags array<string>,
    properties map<string, string>
"""
events_m = spark.read.schema(schema).json(f"{DATA}/events.jsonl")
events_m.select(
    "properties",
    F.col("properties")["page"].alias("page"),
    F.map_keys("properties").alias("keys"),
).show(3, truncate=False)

events_m.select("user_id", F.explode("properties").alias("key", "value")).show(4)   # map → key/value rows

## Building complex columns

In [ ]:
events.select(
    F.struct("event", "user_id").alias("as_struct"),
    F.array(F.lit(1), F.lit(2), F.lit(3)).alias("as_array"),
    F.create_map(F.lit("os"), F.col("device.os")).alias("as_map"),
).show(2, truncate=False)

# and the reverse of explode: collect back into arrays
events.groupBy("user_id").agg(F.collect_set("event").alias("events_seen")).show(3, truncate=False)

## JSON strings inside columns
Sometimes a column holds raw JSON **text** (common with Kafka messages or database dumps).
* `from_json(col, schema)` — parse it into a struct
* `to_json(col)` — serialise a struct/array/map back to a string
* `get_json_object(col, "$.path")` — pull out one field without a schema

In [ ]:
raw = spark.createDataFrame([('{"name": "Ann", "langs": ["py", "sql"]}',),
                             ('{"name": "Ben", "langs": ["scala"]}',)], "payload string")
parsed = raw.select(F.from_json("payload", "name string, langs array<string>").alias("p"))
parsed.select("p.name", "p.langs").show()
raw.select(F.get_json_object("payload", "$.name").alias("name")).show()
events.select(F.to_json("device").alias("device_json")).show(2, truncate=False)

## Higher-order functions: transform arrays without exploding
Pass a Python `lambda` that receives Spark Columns:

In [ ]:
nums = spark.createDataFrame([([1, 2, 3, 4],), ([10, 20],)], "xs array<int>")
nums.select(
    "xs",
    F.transform("xs", lambda x: x * 10).alias("times10"),
    F.filter("xs", lambda x: x % 2 == 0).alias("evens"),
    F.exists("xs", lambda x: x > 15).alias("any_gt_15"),
    F.aggregate("xs", F.lit(0), lambda acc, x: acc + x).alias("total"),
).show()

### Exercise 1
Create `events_per_os`: columns **`os`, `n`** — the number of events per device OS.

In [ ]:
events_per_os = None  # TODO

In [ ]:
# Check — run me after your answer
assert sorted(events_per_os.columns) == ["n", "os"]
assert {r.os: r.n for r in events_per_os.collect()} == {'iOS': 508, 'Android': 505, 'Windows': 502, 'macOS': 485}
print("Correct!")

### Exercise 2
Create `tag_counts`: columns **`tag`, `n`** — how many times each tag appears across all events.

In [ ]:
tag_counts = None  # TODO

In [ ]:
# Check — run me after your answer
assert {r.tag: r.n for r in tag_counts.collect()} == {'promo': 471, 'organic': 459, 'email': 451, 'new': 448, 'mobile': 444, 'sale': 420}
print("Correct!")

### Exercise 3
Store in `no_tags` the number of events whose `tags` array is **empty**.

In [ ]:
no_tags = None  # TODO

In [ ]:
# Check — run me after your answer
assert no_tags == 391, no_tags
print("Correct!")

### Exercise 4
Parse `orders_json` (below is already in the cell) into `parsed_orders` with **flat** columns **`sku`, `qty`, `w`, `h`** (`qty`, `w`, `h` as ints).

In [ ]:
orders_json = spark.createDataFrame([
    ('{"sku": "A1", "qty": 2, "dims": {"w": 10, "h": 5}}',),
    ('{"sku": "B7", "qty": 1, "dims": {"w": 3, "h": 8}}',),
], "payload string")

parsed_orders = None  # TODO

In [ ]:
# Check — run me after your answer
assert parsed_orders.columns == ["sku", "qty", "w", "h"]
assert [tuple(r) for r in parsed_orders.collect()] == [("A1", 2, 10, 5), ("B7", 1, 3, 8)]
print("Correct!")

### Exercise 5
Create `upper_tags`: `events` plus a column **`tags_upper`** with every tag upper-cased — **without** exploding (use `F.transform`).

In [ ]:
upper_tags = None  # TODO

In [ ]:
# Check — run me after your answer
r = upper_tags.filter(F.size("tags") > 1).first()
assert r.tags_upper == [t.upper() for t in r.tags]
print("Correct!")

---
Stuck? Worked answers (with outputs) are in [`../solutions/08_complex_types.ipynb`](../solutions/08_complex_types.ipynb).